# 02 · Réplica de la propuesta de alertas y tabla integrada

Reto 1 — Monitoreo de saldo de inventarios, FEMSA Salud Chile (FCV).

El área de Auditoría Interna compartió un archivo de Excel (*Propuesta de alertas inventarios*) donde construye, para julio de 2026, las columnas de alerta del Kardex y un dashboard con la conciliación contra la balanza. Este notebook hace dos cosas:

1. Verifica que el código de `src/alertas_inventario.py` reproduce los resultados de ese archivo, columna por columna y en cada indicador del dashboard.
2. Aplica el mismo código a enero para obtener la tabla integrada Kardex + Balanza con la que se trabajará, y resume los siete meses disponibles.

Requisito: Excel desencriptados en `data/raw/` (`dvc pull -r storage` y `./scripts/dvc_decrypt.sh`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)


def raiz_del_repo():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise FileNotFoundError("No se encontró pyproject.toml; abrir el notebook dentro del repo")


ROOT = raiz_del_repo()
RAW = ROOT / "data" / "raw"
FIGURAS = ROOT / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
import alertas_inventario as ai
from construir_columnas import ubicar_archivos

archivos = ubicar_archivos(RAW)
PROPUESTA = next(RAW.glob("*Propuesta*alertas*.xlsx"))
list(archivos), PROPUESTA.name

## 1. Fórmulas traducidas

Cada columna calculada del Kardex en el Excel tiene su función en `src/alertas_inventario.py`:

| Columna Excel | Fórmula en Excel | Función |
|---|---|---|
| VALORIZACION (F) | `=E*D` | `valorizacion` |
| MESES ANTIG. (I) | `=IF(G="","",MAX(0,(YEAR(EOMONTH(A,0))-YEAR(G))*12+MONTH(EOMONTH(A,0))-MONTH(G)))` | `meses_antiguedad` |
| ALERTA ANTIG. (J) | `=IF(G="","SIN FECHA",IF(I>12,"ROJO",IF(I>6,"ÁMBAR","VERDE")))` | `alerta_antiguedad` |
| DIF. VALORIZACIÓN (K) | `=F-(D*E)` | `dif_valorizacion` |
| ALERTA STOCK (L) | `=IF(D<0,"NEGATIVO",IF(D=0,"CERO","OK"))` | `alerta_stock` |
| ALERTA COSTO (M) | `=IF(E<=0,"INVÁLIDO","OK")` | `alerta_costo` |
| ALERTA CONVERSIÓN (N) | `=IF(OR(H="",H<=0),"INVÁLIDA",IF(H<>1,"REVISAR","OK"))` | `alerta_conversion` |
| ALERTA DUPLICADO* (O) | valor fijo `OK` | `alerta_duplicado` (COUNTIF > 1) |
| ESTADO GENERAL (P) | `=IF(OR(J="ROJO",L<>"OK",M<>"OK",ABS(K)>1,N<>"OK",O<>"OK"),"REVISAR",IF(J="ÁMBAR","MONITOREAR","OK"))` | `estado_general` |

Y en el dashboard:

| Indicador | Excel | Función |
|---|---|---|
| Saldo GL | `XLOOKUP("CTA_000184", balanza!A, balanza!K)` | `saldo_gl` |
| Inventario conciliable | total detalle Kardex + `'TD DETA'!F` (recetario) | `conciliacion` |
| Estado de conciliación | `IF(ABS(dif)<=ABS(GL)*0.001,"CONCILIADO","DIFERENCIA")` | `conciliacion` |
| Antigüedad por color | `COUNTIF` / `SUMIF` sobre la columna J | `resumen_antiguedad` |
| Otras alertas | `COUNTIF` / `SUMIF` sobre L, M, N, O, P | `resumen_otras_alertas` |
| Top 10 rojas | `LARGE` + `XLOOKUP` | `top_rojas` |
| Hoja *Alertas Inventario* | filtro de REVISAR y MONITOREAR | `detalle_alertas` |

Hay dos diferencias deliberadas con el Excel:

- En la propuesta, `ALERTA DUPLICADO*` es un texto fijo; aquí se calcula, como la misma propuesta recomienda.
- Cuando `UNID CONV` trae `#N/A`, Excel propaga el error y la partida queda sin estado. Aquí se marca como `INVÁLIDA` y pasa a `REVISAR`, para que no se pierda.

## 2. Réplica de julio

Se procesa el Kardex y la balanza de julio con el código y se compara contra la hoja `DETALLE KARDEX Julio` de la propuesta, que tiene los valores calculados por las fórmulas de Excel.

In [ ]:
kardex_jul, balanza_jul = archivos["202607"]
detalle_jul, resumen_jul = ai.procesar_mes(kardex_jul, balanza_jul)

excel = pd.read_excel(PROPUESTA, sheet_name="DETALLE KARDEX Julio ")
excel = excel[excel["CODIGO"].notna()].rename(columns={
    "MESES ANTIG.": "MESES_ANTIG", "ALERTA ANTIG.": "ALERTA_ANTIG",
    "DIF. VALORIZACIÓN": "DIF_VALORIZACION", "ALERTA STOCK": "ALERTA_STOCK",
    "ALERTA COSTO": "ALERTA_COSTO", "ALERTA CONVERSIÓN": "ALERTA_CONVERSION",
    "ALERTA DUPLICADO*": "ALERTA_DUPLICADO", "ESTADO GENERAL": "ESTADO_GENERAL",
})

print(f"Partidas en el código: {len(detalle_jul):,} | en el Excel: {len(excel):,}")
comp = detalle_jul.merge(excel, on="CODIGO", suffixes=("_py", "_xl"), validate="one_to_one")
print(f"Partidas emparejadas por CODIGO: {len(comp):,}")

In [ ]:
def iguales(a, b):
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        return np.isclose(a.astype(float), b.astype(float), atol=0.01, equal_nan=True)
    return a.astype(str) == b.astype(str)

columnas = ["VALORIZACION", "MESES_ANTIG", "ALERTA_ANTIG", "DIF_VALORIZACION", "ALERTA_STOCK",
            "ALERTA_COSTO", "ALERTA_CONVERSION", "ALERTA_DUPLICADO", "ESTADO_GENERAL"]

coincidencia = pd.DataFrame({
    "coinciden": [int(iguales(comp[f"{c}_py"], comp[f"{c}_xl"]).sum()) for c in columnas],
    "no_coinciden": [int((~iguales(comp[f"{c}_py"], comp[f"{c}_xl"])).sum()) for c in columnas],
}, index=columnas)
coincidencia["pct"] = coincidencia["coinciden"] / len(comp) * 100
coincidencia

Todas las columnas coinciden en el 100% de las partidas, salvo `ALERTA_CONVERSION` y `ESTADO_GENERAL` en dos partidas. Son las dos que traen `#N/A` en la unidad de conversión:

In [ ]:
diferentes = ~iguales(comp["ESTADO_GENERAL_py"], comp["ESTADO_GENERAL_xl"])
comp.loc[diferentes, ["CODIGO", "UNID_CONV", "UNID CONV", "ALERTA_CONVERSION_py", "ALERTA_CONVERSION_xl",
                      "ESTADO_GENERAL_py", "ESTADO_GENERAL_xl"]]

En el Excel quedan vacías (error propagado) y en el código quedan como `INVÁLIDA` / `REVISAR`. La propia hoja de criterios de la propuesta recomienda mantener visibles estas filas, así que es la diferencia deliberada descrita en la sección 1.

### Indicadores del dashboard

Se leen los valores que muestra la hoja *Dashboard Inventario Julio* y se comparan con los del código.

In [ ]:
ws = openpyxl.load_workbook(PROPUESTA, read_only=True, data_only=True)["Dashboard Inventario Julio"]
celda = {(i, j): v for i, fila in enumerate(ws.iter_rows(max_row=40, values_only=True), start=1)
         for j, v in enumerate(fila, start=1)}

antig = ai.resumen_antiguedad(detalle_jul, resumen_jul["INVENTARIO_CONCILIABLE"])
otras = ai.resumen_otras_alertas(detalle_jul)

filas = [
    ("Saldo GL CTA_000184", celda[5, 1], resumen_jul["SALDO_GL"]),
    ("Inventario conciliable (Kardex + recetario)", celda[5, 4], resumen_jul["INVENTARIO_CONCILIABLE"]),
    ("Diferencia Kardex − GL", celda[5, 7], resumen_jul["DIFERENCIA"]),
    ("Estado de conciliación", celda[5, 10], resumen_jul["ESTADO_CONCILIACION"]),
    ("SKU únicos", celda[5, 12], resumen_jul["SKUS"]),
]
for fila_xl, color in zip(range(14, 18), ["ROJO", "ÁMBAR", "VERDE", "SIN FECHA"]):
    filas.append((f"Partidas {color}", celda[fila_xl, 2], antig.loc[color, "PARTIDAS"]))
    filas.append((f"Valorización {color}", celda[fila_xl, 3], antig.loc[color, "VALORIZACION"]))
for fila_xl, alerta in zip(range(14, 21), otras.index):
    filas.append((f"{alerta} (partidas)", celda[fila_xl, 6], otras.loc[alerta, "PARTIDAS"]))
    filas.append((f"{alerta} (valor)", celda[fila_xl, 7], otras.loc[alerta, "VALORIZACION"]))

dashboard = pd.DataFrame(filas, columns=["indicador", "excel", "python"])
dashboard["coincide"] = [
    (np.isclose(float(x), float(p), atol=0.01) if isinstance(x, (int, float)) else str(x) == str(p))
    for x, p in zip(dashboard["excel"], dashboard["python"])
]
dashboard

El único indicador que no coincide es *Total para revisar*: el código cuenta dos partidas más (las de `#N/A`, por 26,000 CLP). En *Conversión distinta de 1 / inválida* el conteo sí coincide (109), porque el `COUNTIF(N,"<>OK")` de Excel también cuenta las celdas con error.

In [ ]:
top_xl = pd.DataFrame([[celda[f, c] for c in range(1, 5)] for f in range(23, 33)],
                      columns=["CODIGO", "DESCRIPTOR", "MESES_ANTIG", "VALORIZACION"])
top_py = ai.top_rojas(detalle_jul)
print("Top 10 rojas idéntico:", top_xl["CODIGO"].tolist() == top_py["CODIGO"].tolist()
      and np.allclose(top_xl["VALORIZACION"], top_py["VALORIZACION"]))
top_py

In [ ]:
alertas_xl = pd.read_excel(PROPUESTA, sheet_name="Alertas Inventario", skiprows=3).dropna(subset=["Código"])
alertas_py = ai.detalle_alertas(detalle_jul)
solo_py = set(alertas_py["CODIGO"]) - set(alertas_xl["Código"])
solo_xl = set(alertas_xl["Código"]) - set(alertas_py["CODIGO"])
print(f"Hoja Alertas Inventario: {len(alertas_xl):,} partidas | código: {len(alertas_py):,}")
print("Solo en el código:", sorted(solo_py), "| solo en Excel:", sorted(solo_xl))

La lista de partidas a revisar o monitorear es la misma, más las dos partidas con `#N/A`. Con esto el código reproduce la propuesta de julio.

## 3. Tabla integrada de enero

Con las fórmulas validadas, se aplica el mismo proceso a enero: se cruza el Kardex con la cuenta de existencias de la balanza y con el recetario del mes.

In [ ]:
detalle_ene, resumen_ene = ai.procesar_mes(*archivos["202601"])
print(f"{len(detalle_ene):,} partidas, {detalle_ene.shape[1]} columnas")
detalle_ene.head()

In [ ]:
pd.Series({k: resumen_ene[k] for k in ["SALDO_GL", "TOTAL_KARDEX", "RECETARIO", "INVENTARIO_CONCILIABLE",
                                        "DIFERENCIA", "ESTADO_CONCILIACION", "SKUS"]}, name="enero 2026")

In [ ]:
ai.resumen_antiguedad(detalle_ene, resumen_ene["INVENTARIO_CONCILIABLE"])

In [ ]:
ai.resumen_otras_alertas(detalle_ene)

In [ ]:
detalle_ene["ESTADO_GENERAL"].value_counts()

En enero la conciliación cuadra dentro de la tolerancia (diferencia de −0.43 CLP sobre 118,887 millones). Hay 2,140 partidas en rojo por 171.6 millones CLP y 2,216 partidas en estado REVISAR, la mayoría por antigüedad y el resto por unidad de conversión distinta de 1. Por línea, Mercaderías Varias destaca: es la de menor valor, pero su mediana de antigüedad es de 45 meses y 269 de sus 395 partidas están en rojo.

### Estadística descriptiva (enero)

In [ ]:
desc = detalle_ene[["STOCK", "COSTO", "VALORIZACION", "MESES_ANTIG"]].astype(float).describe(
    percentiles=[0.25, 0.5, 0.75]).T
desc["coef_var"] = desc["std"] / desc["mean"]
desc

In [ ]:
por_linea = (detalle_ene.groupby("NOMBRE_LINEA")
             .agg(PARTIDAS=("CODIGO", "count"), VALORIZACION=("VALORIZACION", "sum"),
                  MESES_MEDIANA=("MESES_ANTIG", "median"),
                  PARTIDAS_ROJO=("ALERTA_ANTIG", lambda s: (s == "ROJO").sum()))
             .sort_values("VALORIZACION", ascending=False))
por_linea["PCT_VALOR"] = por_linea["VALORIZACION"] / por_linea["VALORIZACION"].sum() * 100
por_linea

In [ ]:
detalle_ene.groupby("POLITICA_CANJE")["ALERTA_ANTIG"].value_counts().unstack(fill_value=0)

## 4. Los siete meses

El mismo proceso para enero a julio da una fila de indicadores por mes. Aquí entra también la regla de variación mensual (alerta si el saldo cambia 5% o más respecto al cierre anterior).

In [ ]:
resumenes = [ai.procesar_mes(*archivos[mes])[1] for mes in archivos]
mensual = ai.variacion_mensual(pd.DataFrame(resumenes))
mensual[["PERIODO", "SALDO_GL", "TOTAL_KARDEX", "RECETARIO", "DIFERENCIA", "ESTADO_CONCILIACION",
         "VARIACION_PCT", "ALERTA_VARIACION", "PARTIDAS_ROJO", "PARTIDAS_AMBAR", "PARTIDAS_REVISAR",
         "PARTIDAS_FULCOM_FUTURA", "PARTIDAS_VALOR_NEGATIVO", "GL_SALDO_ACREEDOR"]]

- **Conciliación:** los siete meses concilian con la tolerancia de 0.1%. La mayor diferencia es abril (19.72 CLP); con un umbral estricto de 0% como el del planteamiento, ningún mes cuadraría exactamente por diferencias de centavos y redondeo.
- **Variación:** solo mayo dispara la alerta (−7.35%).
- **Naturaleza contraria:** no hay partidas con valorización negativa ni saldo acreedor en la cuenta de existencias.
- **Fechas futuras:** entre 10 y 11 mil partidas por mes tienen FULCOM posterior al cierre. La fórmula de antigüedad las lleva a 0 meses, es decir, a verde.

### Figuras para el reporte

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8a85", "axes.labelcolor": "#3d3d3a",
                     "xtick.color": "#5f5e5a", "ytick.color": "#5f5e5a"})
AZUL, CRITICO, ADVERTENCIA = "#2a78d6", "#d03b3b", "#fab219"
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
meses_txt = [MESES_ES[m - 1] for m in mensual["PERIODO"].dt.month]

fig, ax = plt.subplots(figsize=(6.2, 2.8))
saldo = mensual["SALDO_GL"] / 1e9
x = np.arange(len(meses_txt))
ax.plot(x, saldo, color=AZUL, linewidth=2, zorder=2)
alerta = (mensual["ALERTA_VARIACION"] == "ALERTA").to_numpy()
ax.scatter(x[~alerta], saldo[~alerta], s=36, color=AZUL, edgecolor="white", linewidth=1.5, zorder=3)
ax.scatter(x[alerta], saldo[alerta], s=48, color=CRITICO, edgecolor="white", linewidth=1.5, zorder=3,
           label="Variación ≥ 5% (alerta)")
for i, (v, var) in enumerate(zip(saldo, mensual["VARIACION_PCT"])):
    if pd.notna(var):
        dy, va = (0.9, "bottom") if var >= 0 else (-0.9, "top")
        ax.text(i, v + dy, f"{var:+.1%}", ha="center", va=va, fontsize=8, color="#3d3d3a")
ax.set_xticks(x, meses_txt)
ax.set_yticks(range(110, 135, 5))
ax.set_ylim(110, 132)
ax.set_ylabel("Saldo GL, miles de millones CLP")
ax.yaxis.grid(True, color="#e4e3dd", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower left")
fig.tight_layout()
fig.savefig(FIGURAS / "saldo_gl_mensual.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 2.8))
ax.plot(meses_txt, mensual["PARTIDAS_ROJO"], color=CRITICO, linewidth=2, marker="o", markersize=5, label="Rojo (> 12 meses)")
ax.plot(meses_txt, mensual["PARTIDAS_AMBAR"], color=ADVERTENCIA, linewidth=2, marker="s", markersize=5, label="Ámbar (7–12 meses)")
ax.text(len(meses_txt) - 0.9, mensual["PARTIDAS_ROJO"].iloc[-1], "Rojo", va="center", fontsize=8)
ax.text(len(meses_txt) - 0.9, mensual["PARTIDAS_AMBAR"].iloc[-1], "Ámbar", va="center", fontsize=8)
ax.set_xlim(-0.4, len(meses_txt) - 0.2)
ax.set_ylim(0, 2400)
ax.set_ylabel("Partidas")
ax.yaxis.grid(True, color="#e4e3dd", linewidth=0.6)
ax.legend(frameon=False, loc="lower left", ncol=2)
fig.tight_layout()
fig.savefig(FIGURAS / "partidas_semaforo_mensual.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 2.8))
datos = por_linea.sort_values("VALORIZACION")
ax.barh(datos.index, datos["VALORIZACION"] / 1e9, color=AZUL, height=0.6)
for i, v in enumerate(datos["VALORIZACION"] / 1e9):
    ax.text(v + 0.4, i, f"{v:,.1f}", va="center", fontsize=8, color="#3d3d3a")
ax.set_xlabel("Valorización, miles de millones CLP (enero 2026)")
ax.xaxis.grid(True, color="#e4e3dd", linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURAS / "valor_por_linea_enero.pdf")
plt.show()

## Conclusión

El módulo `alertas_inventario` reproduce las columnas y el dashboard de la propuesta de julio, con dos diferencias intencionales (duplicados calculados y partidas con `#N/A` visibles). El script `src/construir_columnas.py` aplica el mismo proceso a cualquier mes y genera `kardex_alertas.csv` y `resumen_mensual.csv` en `data/processed/`, que son la base para el dashboard en Power BI y para las matrices de transición del modelo de Markov.